In [ ]:

# ============================================================
# VGG16 ABLATION STUDY - KAGGLE READY SCRIPT
# Author: Rofiq
# ============================================================

# ===================== CELL 1: IMPORTS =====================

import os
import copy
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

# ===================== CELL 2: CONFIG =======================

DATASET_PATH = "/kaggle/input/YOUR_DATASET_FOLDER"  # CHANGE THIS
BATCH_SIZE = 32
IMG_SIZE = 224
EPOCHS = 25
LR = 1e-4
PATIENCE = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", device)

# ===================== CELL 3: TRANSFORM ====================

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ===================== CELL 4: DATASET ======================

dataset = datasets.ImageFolder(DATASET_PATH, transform=transform)
class_names = dataset.classes
NUM_CLASSES = len(class_names)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Classes:", class_names)

# ===================== CELL 5: CLASS WEIGHTS ================

targets = [label for _, label in train_dataset]
class_counts = np.bincount(targets)
class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# ===================== CELL 6: EARLY STOPPING ===============

class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.best_loss = np.inf
        self.counter = 0
        self.best_model = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            self.best_model = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
        return self.counter >= self.patience

# ===================== CELL 7: MODEL BUILDER ================

def build_vgg16(mode="freeze_all"):
    model = models.vgg16(pretrained=True)

    if mode == "freeze_all":
        for param in model.features.parameters():
            param.requires_grad = False

    elif mode == "unfreeze_last_block":
        for param in model.features.parameters():
            param.requires_grad = False
        for param in model.features[24:].parameters():
            param.requires_grad = True

    elif mode == "full_finetune":
        for param in model.features.parameters():
            param.requires_grad = True

    model.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    return model.to(device)

# ===================== CELL 8: TRAIN FUNCTION ===============

def train_model(model, mode_name):

    optimizer = optim.AdamW(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5
    )

    early_stopping = EarlyStopping(patience=PATIENCE)

    for epoch in range(EPOCHS):

        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                preds = torch.argmax(outputs, 1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss /= len(val_loader)
        scheduler.step(val_loss)

        print(f"{mode_name} | Epoch {epoch+1} | Val Loss: {val_loss:.4f}")

        if early_stopping.step(val_loss, model):
            print("Early Stopping Triggered")
            break

    model.load_state_dict(early_stopping.best_model)

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="weighted"
    )

    return acc, precision, recall, f1

# ===================== CELL 9: ABLATION RUN =================

modes = ["freeze_all", "unfreeze_last_block", "full_finetune"]
results = []

for mode in modes:
    print(f"\nRunning Ablation: {mode}\n")
    model = build_vgg16(mode)
    acc, prec, rec, f1 = train_model(model, mode)
    results.append([mode, acc, prec, rec, f1])

# ===================== CELL 10: SAVE RESULTS ================

with open("ablation_results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Mode", "Accuracy", "Precision", "Recall", "F1-Score"])
    writer.writerows(results)

print("\nAblation Study Complete ✅")
print("Results saved to ablation_results.csv")